# **Coin Detection, Classification, and Counting**
## Image Processing and Computer Vision – Assignment Module #1


## Task
Given a dataset of photographs containing coins, detect all coins present in each image, identify their denomination, and compute the total monetary value. **Every step uses traditional Computer Vision techniques.**

Expected output format:
```
image_85.jpg - 5 coin(s) found:
  Coin 1 {value: 1.000}
  ...
  Partial Amount {value: 3.350}
...
Total Amount: TBD €
```

### Techniques used and where they come from
| Step | Technique | Source |
|---|---|---|
| Coin localization | `HoughCircles` | L1 / L5 (Hough Transform) |
| Pre-Hough smoothing | Median filter | L2 (slides 25–26) |
| Candidate verification | Sobel gradient | L3 (Prewitt and Sobel, slide 12) |
| Duplicate removal | Non-Maxima Suppression | L3 (NMS, slides 14–15) |
| Scale-consistency filter | Euro coin diameters (geometry) | physical prior, see below |
| Coin classification | SIFT descriptors | L4 |
| Match filtering | Lowe's ratio test | L4 / Lab 2 |
| Geometric verification | `findHomography` + RANSAC | L4 / Lab 2 |
| Match visualization | `drawMatches` | Lab 2 |

No technique outside the course syllabus is used (in particular **no histogram equalization / CLAHE**, which is not part of the course material).

## ⚠️ Current state: detection works, recognition does not

**This is the single most important thing to know about the current pipeline.** The two sub-problems are in very different shape, and they must be read separately:

| Sub-problem | Status |
|---|---|
| **Counting** (how many coins are in the image) | **90.1% exact** — works reasonably |
| **Recognition** (which denomination each coin is) | **essentially failing** — see below |

### Why recognition is failing
A homography is determined by **exactly 4 point correspondences**. This means `cv2.findHomography` with RANSAC returns at least 4 inliers *by construction*, even for completely random matches. A `min_inliers=4` acceptance threshold therefore accepts pure noise.

Measured evidence:

| | RANSAC inliers |
|---|---|
| True match (each reference classified against the reference set) | **206 – 500** |
| Best *wrong* candidate in the same test | 6 – 12 |
| Best candidate on actual target ROIs | **median 8, max 35** |

Every target coin scores in the noise band, so the denomination assigned is effectively arbitrary. Spot checks on coins identified by eye confirm it (e.g. a 10 cent coin whose true class collects 4 inliers and loses to `2cent`, also on 4).

Contributing factors: the target photos are strongly degraded (blur + noise destroy the fine engraving SIFT relies on), and many target coins show the **national face** while the reference set only covers the common/value face, so for those coins no correspondence can exist at all.

### Consequence for the metric
`GROUND_TRUTH_RAW` only contains coin **counts**, so the accuracy reported at the bottom of this notebook measures counting only. It is *not* evidence that recognition works, and the printed `Total Amount` is not currently meaningful. Validating recognition requires per-coin denomination labels, which we do not yet have.

## Data
- `reference_set`: eight images, one per denomination, each with a single coin; the file name encodes the class.
- `target_set`: unlabeled images with one or more coins.

Update `ref_dir` / `target_dir` if your folder layout differs.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re

plt.rcParams['figure.figsize'] = (10, 8)

ref_dir = 'coin_dataset/reference_set/'
target_dir = 'coin_dataset/target_set/'

COIN_VALUES = {
    '1cent': 0.01, '2cent': 0.02, '5cent': 0.05,
    '10cent': 0.10, '20cent': 0.20, '50cent': 0.50,
    '1euro': 1.00, '2euro': 2.00
}

# Official euro coin diameters in mm - used by the scale-consistency filter
COIN_DIAMETERS_MM = {
    '1cent': 16.25, '2cent': 18.75, '5cent': 21.25,
    '10cent': 19.75, '20cent': 22.25, '50cent': 24.25,
    '1euro': 23.25, '2euro': 25.75
}
MAX_COIN_RATIO = max(COIN_DIAMETERS_MM.values()) / min(COIN_DIAMETERS_MM.values())  # ~1.585

reference_images = {name: os.path.join(ref_dir, f"{name}.jpg") for name in COIN_VALUES.keys()}

print(f"Largest/smallest euro coin diameter ratio: {MAX_COIN_RATIO:.3f}")
print("Section complete: libraries and constants ready.")

## Ground Truth
Coin **count** per target image (see the warning above: no denomination labels are available). `HARD_CASES` flags images with touching coins, cluttered backgrounds or poor lighting.

In [ ]:
GROUND_TRUTH_RAW = {
    1: 3, 2: 1, 3: 6, 4: 1, 5: 1, 6: 1, 7: 3, 8: 1, 9: 1, 10: 1,
    11: 2, 12: 2, 13: 1, 14: 1, 15: 1, 16: 2, 17: 3, 18: 3, 19: 1, 20: 3,
    21: 1, 22: 1, 23: 2, 24: 3, 25: 3, 26: 2, 27: 5, 28: 2, 29: 1, 30: 3,
    31: 2, 32: 3, 33: 1, 34: 1, 35: 1, 36: 2, 37: 3, 38: 7, 39: 2, 40: 5,
    41: 1, 42: 1, 43: 2, 44: 2, 45: 3, 46: 3, 47: 3, 48: 1, 49: 3, 50: 2,
    51: 4, 52: 6, 53: 1, 54: 2, 55: 2, 56: 1, 57: 3, 58: 3, 59: 2, 60: 1,
    61: 1, 62: 2, 63: 3, 64: 1, 65: 1, 66: 3, 67: 2, 68: 1, 69: 1, 70: 1,
    71: 1, 72: 3, 73: 3, 74: 2, 75: 2, 76: 4, 77: 3, 78: 3, 79: 5, 80: 2,
    81: 1, 82: 1, 83: 1, 84: 4, 85: 5, 86: 3, 87: 1, 88: 6, 89: 2, 90: 2,
    91: 1, 92: 3, 93: 1, 94: 3, 95: 3, 96: 8, 97: 4, 98: 4, 99: 1, 100: 1,
    101: 3, 102: 3, 103: 3, 104: 1, 105: 1, 106: 3, 107: 1, 108: 1, 109: 2, 110: 2,
    111: 3, 112: 3, 113: 1, 114: 1, 115: 4, 116: 1, 117: 1, 118: 8, 119: 1, 120: 2,
    121: 4, 122: 3, 123: 1, 124: 3, 125: 2, 126: 3, 127: 2, 128: 3, 129: 1, 130: 2,
    131: 1, 132: 4, 133: 3, 134: 2, 135: 1, 136: 2, 137: 3, 138: 3, 139: 2, 140: 2,
    141: 1, 142: 2,
}

HARD_CASES = {11, 24, 27, 32, 36, 40, 43, 55, 79, 85, 103, 132}

print(f"Section complete: Ground Truth loaded for {len(GROUND_TRUTH_RAW)} images.")

## Detection — stage 1: HoughCircles

A median filter (L2) removes speckle before the Hough accumulator, which is sensitive to isolated noisy edges. Hough parameters are kept deliberately permissive: it is easier to reject false circles afterwards with explicit, justifiable criteria than to tune `param2` so tightly that real coins in the harder images are lost.

A sweep over `param2` (20/22/24/26/28/30) confirmed 28 as the best operating point — lowering it produced more false circles at radii the later filters cannot distinguish, without recovering the missed coins.

## Detection — stage 2: radial-gradient verification (Sobel)

On textured backgrounds (concrete, gravel, fabric, wicker) Hough hallucinates circles wherever enough curved edges accumulate votes. The check below rejects them:

1. Sample N points on the candidate circumference and compute the gradient with the **Sobel operator** (L3).
2. A genuine coin boundary has the gradient **radially aligned** (pointing in/out of the center) at nearly every sample; background texture has gradient directions uncorrelated with the circle geometry.
3. Keep circles whose fraction of strong *and* radially-aligned samples exceeds `support_thresh`.

*Rejected alternative:* checking only whether *any* Canny edge lies near the circumference, ignoring direction. It was useless in practice — on textured backgrounds an edge is almost always nearby, so scores clustered in 0.85–1.0 for true and false circles alike. The gradient **direction** is what carries the discriminative information.

In [ ]:
def circle_radial_gradient_support(gray, x, y, r, n_samples=72, angle_tol_deg=25, grad_thresh=15):
    """Fraction of points sampled on circle (x, y, r) where the image gradient is both strong
    (>= grad_thresh) AND aligned within angle_tol_deg of the expected radial direction."""
    h, w = gray.shape
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    hits = 0
    valid_samples = 0
    for i in range(n_samples):
        theta = 2 * np.pi * i / n_samples
        px = int(round(x + r * np.cos(theta)))
        py = int(round(y + r * np.sin(theta)))
        if not (0 <= px < w and 0 <= py < h):
            continue
        valid_samples += 1
        mag = np.hypot(gx[py, px], gy[py, px])
        if mag < grad_thresh:
            continue
        grad_angle = np.degrees(np.arctan2(gy[py, px], gx[py, px]))
        radial_angle = np.degrees(theta)
        diff = abs((grad_angle - radial_angle + 180) % 360 - 180)
        diff = min(diff, abs(diff - 180))  # gradient may point inward or outward
        if diff <= angle_tol_deg:
            hits += 1
    return hits / valid_samples if valid_samples else 0.0

print("Section complete: radial-gradient verification ready.")

## Detection — stage 3: Non-Maxima Suppression
Hough frequently returns several near-duplicate circles on the same coin, especially where coins touch. NMS (L3) keeps, for each cluster of overlapping circles, only the one with the best radial-gradient support.

In [ ]:
def non_max_suppress_circles(circles, scores, overlap_ratio=0.6):
    """Keeps the best-scoring circle among overlapping ones. Two circles count as duplicates
    when the distance between centers is below overlap_ratio * larger_radius."""
    if len(circles) == 0:
        return []
    idxs = np.argsort(-np.array(scores))
    keep = []
    used = np.zeros(len(circles), dtype=bool)
    for i in idxs:
        if used[i]:
            continue
        keep.append(i)
        used[i] = True
        x1, y1, r1 = circles[i]
        for j in idxs:
            if used[j] or j == i:
                continue
            x2, y2, r2 = circles[j]
            if np.hypot(x1 - x2, y1 - y2) < overlap_ratio * max(r1, r2):
                used[j] = True
    return keep

print("Section complete: NMS ready.")

## Detection — stage 4: scale-consistency filter (physical prior)

Inspecting the worst over-counting failures showed a consistent pattern: the spurious circles were **pebbles and speckles of the background**, always much smaller than the coins, and they survived the gradient check because small round grains do have a genuine radial boundary.

There is a hard physical constraint that separates them. Euro coins range from 16.25 mm (1 cent) to 25.75 mm (2 euro), so within a single photograph — where all coins lie on the same plane at the same camera distance — **no two coin radii can differ by more than a factor of ≈1.585**. Any circle much smaller than the largest detected circle therefore cannot be a coin.

A small tolerance (`tol`) absorbs the radius estimation error of Hough and mild perspective effects.

*Note on robustness:* this anchors on the largest circle, which is reliable here because false positives in this dataset are smaller than the coins. It would need revisiting on images containing large circular distractors (plates, cups, lids) — one such case remains in the dataset (a wicker basket, `image_136`).

In [ ]:
def filter_by_radius_consistency(circles, scores, tol=1.10):
    """Drops circles too small to be a coin, given the largest circle detected in the same image.
    Rationale: within one photo all coins share the same scale, and the largest euro coin is only
    ~1.585x the diameter of the smallest one."""
    if not circles:
        return circles, scores
    r_max = max(c[2] for c in circles)
    r_min_allowed = r_max / (MAX_COIN_RATIO * tol)
    keep = [(c, s) for c, s in zip(circles, scores) if c[2] >= r_min_allowed]
    if not keep:
        return circles, scores
    return [k[0] for k in keep], [k[1] for k in keep]

print("Section complete: scale-consistency filter ready.")

## Detection — full routine and parameter choice

Parameters were chosen by grid search over `support_thresh` × `overlap_ratio` × `tol`, evaluated against the ground-truth counts. The selected point sits on a **plateau** (`support_thresh` 0.24–0.25, `tol` 1.05–1.10 all give the same result), not on an isolated peak, which makes it a stable rather than an overfitted choice.

Effect of each stage on exact-count accuracy:

| Configuration | Exact count | Hard cases |
|---|---|---|
| Hough + gradient filter + NMS (`support_thresh=0.30`) | 86.6% | 7/12 |
| \+ scale-consistency filter | 88.7% | 9/12 |
| \+ retuned `support_thresh=0.25` | **90.1%** | **9/12** |

In [ ]:
SUPPORT_THRESH = 0.25   # radial-gradient support required to accept a circle
OVERLAP_RATIO  = 0.60   # NMS duplicate criterion
RADIUS_TOL     = 1.10   # slack on the physical coin-size ratio


def detect_coins(image_path, support_thresh=SUPPORT_THRESH, overlap_ratio=OVERLAP_RATIO,
                  tol=RADIUS_TOL, debug=False):
    """Returns (list of accepted circles, grayscale image, BGR image)."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"ERROR: image not found -> {image_path}")
        return [], None, None

    max_width = 1200
    if img.shape[1] > max_width:
        scale = max_width / img.shape[1]
        img = cv2.resize(img, (int(img.shape[1] * scale), int(img.shape[0] * scale)))

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.medianBlur(gray, 15)          # L2: median filter kills speckle before Hough

    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=40,
                                param1=50, param2=28, minRadius=20, maxRadius=150)
    if circles is None:
        if debug:
            print("[Hough] no circles found.")
        return [], gray, img

    circles = np.round(circles[0, :]).astype("int")
    if debug:
        print(f"[Hough]  raw circles:                {len(circles)}")

    scores = [circle_radial_gradient_support(gray, x, y, r) for (x, y, r) in circles]
    if debug:
        for (x, y, r), s in zip(circles, scores):
            print(f"   candidate (x={x}, y={y}, r={r}): support={s:.2f} -> "
                  f"{'KEEP' if s >= support_thresh else 'reject'}")

    keep_idx = [i for i, s in enumerate(scores) if s >= support_thresh]
    kept = [circles[i] for i in keep_idx]
    kept_scores = [scores[i] for i in keep_idx]
    if debug:
        print(f"[Filter] after radial-gradient check: {len(kept)}")

    nms_idx = non_max_suppress_circles(kept, kept_scores, overlap_ratio)
    kept = [kept[i] for i in nms_idx]
    kept_scores = [kept_scores[i] for i in nms_idx]
    if debug:
        print(f"[NMS]    after duplicate removal:    {len(kept)}")

    kept, kept_scores = filter_by_radius_consistency(kept, kept_scores, tol)
    if debug:
        print(f"[Scale]  after size-consistency:     {len(kept)}")
        if kept:
            print(f"         radii kept: {sorted(int(c[2]) for c in kept)}")

    return kept, gray, img

print("Section complete: detection pipeline ready.")

## Classification: SIFT + Lowe's ratio test + RANSAC

Reference photos are full scenes, so the coin is first located with `HoughCircles` and cropped, which means reference and target features are produced the same way (localize → crop → SIFT).

`contrastThreshold=0.004` (default 0.04) keeps low-contrast keypoints that the degraded target images would otherwise lose; `ratio_thresh=0.80` was selected by sweep (Lab 2's 0.7 rejects far too many genuine matches here).

**`MIN_INLIERS` is the parameter to be honest about.** 4 is the theoretical minimum for a homography and therefore accepts random matches (see the warning at the top). Set it to a value in the 15–20 range to see the *real* recognition rate: most coins become `Unknown`, which is the truthful picture of where this stage stands.

In [ ]:
SIFT_CONTRAST_THRESHOLD = 0.004
RATIO_THRESHOLD = 0.80
MIN_INLIERS = 4    # <-- 4 is meaningless (see warning at top). Try 15-20 for an honest reading.

sift = cv2.SIFT_create(nfeatures=500, contrastThreshold=SIFT_CONTRAST_THRESHOLD)
bf = cv2.BFMatcher()


def crop_reference_coin(gray, margin=1.15):
    """Locates the single coin of a reference photo with HoughCircles and crops around it."""
    blurred = cv2.medianBlur(gray, 15)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=200,
                                param1=50, param2=28, minRadius=50, maxRadius=250)
    if circles is None:
        return gray
    x, y, r = np.round(circles[0, 0]).astype(int)
    r2 = int(r * margin)
    return gray[max(0, y - r2):min(gray.shape[0], y + r2),
                max(0, x - r2):min(gray.shape[1], x + r2)]


reference_data = {}
print("Extracting SIFT descriptors from reference models...")
for name, path in reference_images.items():
    if not os.path.exists(path):
        print(f"  WARNING: file not found for {name} -> {path}")
        continue
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    crop = crop_reference_coin(img)
    kp, des = sift.detectAndCompute(crop, None)
    reference_data[name] = {'kp': kp, 'des': des, 'img': crop}
    print(f"  -> [{name}]: {len(kp)} keypoints (crop {crop.shape[1]}x{crop.shape[0]}).")

print("Section complete: reference descriptors ready.")

In [ ]:
def classify_coin_sift_ransac(coin_roi, reference_data, bf_matcher, mask=None,
                               ratio_thresh=RATIO_THRESHOLD, ransac_reproj_thresh=5.0,
                               min_inliers=MIN_INLIERS, debug=False):
    if coin_roi is None or coin_roi.shape[0] < 20 or coin_roi.shape[1] < 20:
        return "Unknown", 0

    kp_target, des_target = sift.detectAndCompute(coin_roi, mask)
    if des_target is None or len(kp_target) < 4:
        if debug:
            print(f"    [debug] too few keypoints on ROI -> Unknown")
        return "Unknown", 0

    best_name, max_score, best_inliers = "Unknown", 0.0, 0
    if debug:
        print(f"    [debug] {len(kp_target)} keypoints on ROI; matching against every reference:")

    for ref_name, ref_data in reference_data.items():
        matches = bf_matcher.knnMatch(ref_data['des'], des_target, k=2)
        good = [m for pair in matches if len(pair) == 2 for m, n in [pair]
                if m.distance < ratio_thresh * n.distance]

        inliers_count, score = 0, 0.0
        if len(good) >= 4:
            src = np.float32([ref_data['kp'][m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst = np.float32([kp_target[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
            M, mask_ransac = cv2.findHomography(src, dst, cv2.RANSAC, ransac_reproj_thresh)
            if mask_ransac is not None:
                inliers_count = int(np.sum(mask_ransac))
                score = inliers_count / len(ref_data['kp'])
                if score > max_score:
                    max_score, best_name, best_inliers = score, ref_name, inliers_count

        if debug:
            print(f"      {ref_name:>8s}: good={len(good):3d}  inliers={inliers_count:3d}  score={score:.3f}")

    if best_inliers < min_inliers:
        if debug:
            print(f"    [debug] best '{best_name}' has {best_inliers} inliers (< {min_inliers}) -> Unknown")
        return "Unknown", 0

    if debug:
        print(f"    [debug] winner: '{best_name}' ({best_inliers} inliers, score={max_score:.3f})")
    return best_name, best_inliers

print("Section complete: classification function ready.")

## Full pipeline

Detection and recognition are deliberately **decoupled**: the coin count comes from the detection stage, and each detected coin is then classified independently. A coin whose denomination cannot be established is still counted, and reported as `Unknown` rather than being silently dropped or assigned an arbitrary value.

In [ ]:
def process_image(image_path, reference_data, bf_matcher, verbose=True, debug=False):
    if debug:
        print(f"===== DEBUG: {os.path.basename(image_path)} =====")

    circles, gray, img = detect_coins(image_path, debug=debug)
    if gray is None:
        return []

    found = []  # list of (coin_name, value_or_None)
    for (x, y, r) in circles:
        rc = int(r)
        y1, y2 = max(0, y - rc), min(img.shape[0], y + rc)
        x1, x2 = max(0, x - rc), min(img.shape[1], x + rc)
        roi = gray[y1:y2, x1:x2]
        if roi.shape[0] < 20 or roi.shape[1] < 20:
            continue

        mask = np.zeros_like(roi)
        cv2.circle(mask, (x - x1, y - y1), max(1, rc - 4), 255, -1)

        if debug:
            print(f"  -> classifying circle (x={x}, y={y}, r={r}):")
        name, inliers = classify_coin_sift_ransac(roi, reference_data, bf_matcher,
                                                   mask=mask, debug=debug)
        found.append((name, COIN_VALUES.get(name)))

    if verbose:
        fname = os.path.basename(image_path)
        known = [v for _, v in found if v is not None]
        print(f"{fname} - {len(found)} coin(s) found:")
        for i, (name, value) in enumerate(found, start=1):
            if value is None:
                print(f"  Coin {i} {{value: unknown}}")
            else:
                print(f"  Coin {i} {{value: {value:.3f}}}")
        print(f"  Partial Amount {{value: {sum(known):.3f}}}")

    return found

print("Section complete: full pipeline ready.")

## Visual walkthrough — detection stages
Every raw Hough circle, colour-coded by the stage that accepted or rejected it.

In [ ]:
def visualize_detection(image_path, support_thresh=SUPPORT_THRESH,
                         overlap_ratio=OVERLAP_RATIO, tol=RADIUS_TOL):
    img = cv2.imread(image_path)
    max_width = 1200
    if img.shape[1] > max_width:
        s = max_width / img.shape[1]
        img = cv2.resize(img, (int(img.shape[1] * s), int(img.shape[0] * s)))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    circles = cv2.HoughCircles(cv2.medianBlur(gray, 15), cv2.HOUGH_GRADIENT, dp=1.2,
                                minDist=40, param1=50, param2=28, minRadius=20, maxRadius=150)
    out = img.copy()
    if circles is None:
        print("No circles found.")
        return out

    circles = np.round(circles[0, :]).astype("int")
    scores = [circle_radial_gradient_support(gray, x, y, r) for (x, y, r) in circles]
    keep_idx = [i for i, s in enumerate(scores) if s >= support_thresh]
    kept = [circles[i] for i in keep_idx]
    kept_scores = [scores[i] for i in keep_idx]
    nms_local = non_max_suppress_circles(kept, kept_scores, overlap_ratio)
    after_nms_idx = [keep_idx[i] for i in nms_local]
    after_nms = [kept[i] for i in nms_local]
    after_nms_scores = [kept_scores[i] for i in nms_local]
    final, _ = filter_by_radius_consistency(after_nms, after_nms_scores, tol)
    final_set = {(int(c[0]), int(c[1]), int(c[2])) for c in final}

    for i, (x, y, r) in enumerate(circles):
        key = (int(x), int(y), int(r))
        if key in final_set:
            color, label = (0, 255, 0), "coin"
        elif i in after_nms_idx:
            color, label = (255, 0, 255), "too small"      # removed by scale consistency
        elif i in keep_idx:
            color, label = (0, 165, 255), "duplicate"       # removed by NMS
        else:
            color, label = (0, 0, 255), "no radial edge"    # removed by gradient check
        cv2.circle(out, (x, y), r, color, 3)
        cv2.putText(out, label, (x - 30, max(15, y - r - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)

    print(f"raw={len(circles)}  after gradient={len(kept)}  after NMS={len(after_nms)}  final={len(final)}")
    return out


for fname in ['image_44.jpg', 'image_27.jpg']:
    vis = visualize_detection(os.path.join(target_dir, fname))
    plt.figure(figsize=(8, 10))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection stages - {fname}\n"
              "green=accepted  magenta=too small  orange=duplicate  red=no radial edge")
    plt.axis('off')
    plt.show()

## Visual walkthrough — SIFT matching
`cv2.drawMatches` (as in Lab 2) connects the RANSAC-inlier keypoints between the winning reference and the target ROI. These plots are also the clearest illustration of the recognition problem described at the top: genuine matches produce hundreds of consistent correspondences, whereas here only a handful of scattered lines appear.

In [ ]:
def classify_visual(coin_roi, reference_data, bf_matcher, mask=None,
                     ratio_thresh=RATIO_THRESHOLD, ransac_reproj_thresh=5.0):
    info = {'name': 'Unknown', 'inliers': 0, 'kp_target': None, 'ref_kp': None,
            'good_matches': None, 'ransac_mask': None, 'ref_name': None}
    kp_t, des_t = sift.detectAndCompute(coin_roi, mask)
    info['kp_target'] = kp_t
    if des_t is None or len(kp_t) < 4:
        return info
    best = 0.0
    for rn, rd in reference_data.items():
        matches = bf_matcher.knnMatch(rd['des'], des_t, k=2)
        good = [m for p in matches if len(p) == 2 for m, n in [p]
                if m.distance < ratio_thresh * n.distance]
        if len(good) >= 4:
            src = np.float32([rd['kp'][m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst = np.float32([kp_t[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
            M, mk = cv2.findHomography(src, dst, cv2.RANSAC, ransac_reproj_thresh)
            if mk is not None:
                inl = int(np.sum(mk))
                sc = inl / len(rd['kp'])
                if sc > best:
                    best = sc
                    info.update(name=rn, inliers=inl, ref_kp=rd['kp'],
                                good_matches=good, ransac_mask=mk, ref_name=rn)
    return info


def visualize_sift(image_path, max_shown=3):
    circles, gray, img = detect_coins(image_path)
    shown = 0
    for (x, y, r) in circles:
        if shown >= max_shown:
            break
        rc = int(r)
        y1, y2 = max(0, y - rc), min(img.shape[0], y + rc)
        x1, x2 = max(0, x - rc), min(img.shape[1], x + rc)
        roi = gray[y1:y2, x1:x2]
        if roi.shape[0] < 20:
            continue
        mask = np.zeros_like(roi)
        cv2.circle(mask, (x - x1, y - y1), max(1, rc - 4), 255, -1)
        info = classify_visual(roi, reference_data, bf, mask=mask)
        print(f"circle (x={x}, y={y}, r={r}) -> best guess '{info['name']}' "
              f"with only {info['inliers']} inliers")
        if info['good_matches'] is not None and info['ransac_mask'] is not None:
            inl = [m for m, k in zip(info['good_matches'], info['ransac_mask'].ravel()) if k]
            vis = cv2.drawMatches(reference_data[info['ref_name']]['img'], info['ref_kp'],
                                   roi, info['kp_target'], inl, None,
                                   flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
                                   matchColor=(0, 255, 0))
            plt.figure(figsize=(8, 4))
            plt.imshow(vis, cmap='gray')
            plt.title(f"reference '{info['ref_name']}' vs detected coin - "
                      f"{info['inliers']} inliers (a true match scores 200+)")
            plt.axis('off')
            plt.show()
            shown += 1


visualize_sift(os.path.join(target_dir, 'image_27.jpg'))

## Running the full target set

In [ ]:
def numeric_key(fname):
    m = re.search(r'(\d+)', fname)
    return int(m.group(1)) if m else fname

target_files = sorted(
    [f for f in os.listdir(target_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))],
    key=numeric_key
)

all_results = {}
total_amount = 0.0
n_unknown = 0

for fname in target_files:
    coins = process_image(os.path.join(target_dir, fname), reference_data, bf,
                          verbose=True, debug=False)
    all_results[numeric_key(fname)] = coins
    total_amount += sum(v for _, v in coins if v is not None)
    n_unknown += sum(1 for _, v in coins if v is None)

print()
print(f"Total Amount: {total_amount:.3f} \u20ac")
print(f"(coins detected but not recognised: {n_unknown})")

## Evaluation

**Counting** is evaluated against `GROUND_TRUTH_RAW`. **Recognition cannot be evaluated** — no denomination labels exist for the target set — so the figures below describe detection quality only, and must not be read as recognition accuracy.

In [ ]:
correct = hard_correct = 0
over, under = [], []

for n, gt in GROUND_TRUTH_RAW.items():
    found_n = len(all_results.get(n, []))
    if found_n == gt:
        correct += 1
        if n in HARD_CASES:
            hard_correct += 1
    elif found_n > gt:
        over.append(n)
    else:
        under.append(n)

n_images = len(GROUND_TRUTH_RAW)
print("=== COUNTING (evaluable) ===")
print(f"Images with EXACT coin count: {correct}/{n_images} ({100*correct/n_images:.1f}%)")
print(f"Hard cases solved correctly:  {hard_correct}/{len(HARD_CASES)}")
print(f"Over-counted  ({len(over):2d}): {over}")
print(f"Under-counted ({len(under):2d}): {under}")
print()
print("=== RECOGNITION (NOT evaluable) ===")
print("No per-coin denomination labels available for the target set.")
print(f"With MIN_INLIERS={MIN_INLIERS}, coins left as 'Unknown': {n_unknown}")
print("Reminder: a true match scores 200-500 inliers; target ROIs score ~8 (median).")

---
## Next steps

**Recognition (the blocking problem).**
- Obtain denomination labels for a subset of the target set (even 20–30 images). Without them the classifier cannot be developed or validated — only the counting metric can, and optimising it teaches us nothing about recognition.
- Check whether the reference set is meant to cover both coin faces. Target coins shown on the national face have no counterpart in the current reference set, so they are unrecognisable by construction.
- Once labels exist: set `MIN_INLIERS` from the measured inlier distributions of true vs. false matches, rather than from the homography's algebraic minimum.
- Consider exploiting relative radii **within** an image as a complementary cue: absolute size is unusable (measured scale varies from 6.9 to 9.4 px/mm across photos, i.e. camera distance changes), but the ordering of coin sizes inside one photo is informative.

**Counting (working, with known residual failures).**
- Over-counting (4 images) is mostly large circular distractors, e.g. the wicker basket in `image_136`, which defeats the largest-circle anchor of the scale filter. A more robust anchor (weighted radius mode instead of the maximum) would help.
- Under-counting (10 images) is dominated by coins Hough never proposes: in `image_28`, `43`, `54`, `120`, `125` raw Hough returns a single circle where two coins are present. Lowering `param2` did not recover them without adding worse false positives, so these need a different treatment (e.g. searching for missed coins near detected ones, at a compatible radius).